# 03 - Extract Skills from CV

Notebook này dùng để **trích xuất kỹ năng từ CV**.

Hiểu đơn giản:

```text
CV đã tách section
+ danh sách skill chuẩn
+ alias/tên gọi khác của skill
        ↓
quét từng phần của CV
        ↓
tìm skill xuất hiện trong CV
        ↓
03_cv_skills_extracted.xlsx
```

File này **chưa gán skill vào taxonomy**.  
File này chỉ trả lời câu hỏi:

> Trong CV của ứng viên có nhắc đến những skill nào?

## 1. Import thư viện

Các thư viện chính:

- `pandas`: đọc/ghi Excel.
- `re`: xử lý text bằng regular expression.
- `ast`: đọc list nếu list bị lưu thành chuỗi trong Excel.
- `Path`: quản lý đường dẫn.

In [18]:
import ast
import re
from pathlib import Path

import pandas as pd

## 2. Khai báo đường dẫn

Input chính:

- `02_cv_sectioned.xlsx`: CV đã được tách thành các section.
- `12_skill_master.xlsx`: danh sách skill chuẩn.
- `08_djinni_step4_final.xlsx`: file chứa alias/tên gọi mở rộng của skill.

Output:

- `03_cv_skills_extracted.xlsx`: CV kèm danh sách skill đã tìm được.

In [19]:
BASE_DIR = Path("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ProcessPipeline")
DATAXOMY_DIR = Path("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/DataXomy")

CV_SECTION_PATH = BASE_DIR / "data_outputs" / "step02_sections" / "02_cv_sectioned.xlsx"
SKILL_MASTER_PATH = DATAXOMY_DIR / "ESCO_taxonomy" / "notebook_clean" / "12_skill_master.xlsx"
DJINNI_ALIAS_PATH = DATAXOMY_DIR / "Djinni" / "notebook_clean" / "08_djinni_step4_final.xlsx"

OUTPUT_DIR = BASE_DIR / "data_outputs" / "step03_skill_extract"
OUTPUT_PATH = OUTPUT_DIR / "03_cv_skills_extracted.xlsx"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("CV section:", CV_SECTION_PATH, CV_SECTION_PATH.exists())
print("Skill master:", SKILL_MASTER_PATH, SKILL_MASTER_PATH.exists())
print("Djinni alias:", DJINNI_ALIAS_PATH, DJINNI_ALIAS_PATH.exists())
print("Output:", OUTPUT_PATH)

CV section: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ProcessPipeline/data_outputs/step02_sections/02_cv_sectioned.xlsx True
Skill master: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/DataXomy/ESCO_taxonomy/notebook_clean/12_skill_master.xlsx True
Djinni alias: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/DataXomy/Djinni/notebook_clean/08_djinni_step4_final.xlsx True
Output: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ProcessPipeline/data_outputs/step03_skill_extract/03_cv_skills_extracted.xlsx


## 3. Đọc dữ liệu

Notebook đọc 3 bảng:

```text
cv_df     = CV đã tách section
skill_df  = danh sách skill chuẩn
djinni_df = alias/tên gọi khác của skill
```

Mục tiêu của bước này chỉ là nạp dữ liệu vào notebook.

In [20]:
cv_df = pd.read_excel(CV_SECTION_PATH, engine="openpyxl")
skill_df = pd.read_excel(SKILL_MASTER_PATH, engine="openpyxl")
djinni_df = pd.read_excel(DJINNI_ALIAS_PATH, engine="openpyxl")

print("cv_df:", cv_df.shape)
print("skill_df:", skill_df.shape)
print("djinni_df:", djinni_df.shape)

display(cv_df.head(2))
display(skill_df.head(2))
display(djinni_df.head(2))

cv_df: (20, 20)
skill_df: (1172, 10)
djinni_df: (1171, 28)


,candidate_id,cv_text_raw,cv_text_clean,raw_length,clean_length,is_empty_clean,section_summary,section_skills,section_experience,section_education,section_projects,section_certifications,section_other,section_summary_len,section_skills_len,section_experience_len,section_education_len,section_projects_len,section_certifications_len,section_other_len
0,C001,"Frontend developer experienced in ReactJS, Vue...",frontend developer experienced in reactjs vuej...,249,240,False,NaN,NaN,NaN,NaN,NaN,NaN,frontend developer experienced in reactjs vuej...,0,0,0,0,0,0,240
1,C002,"Backend engineer experienced in Java, Spring B...",backend engineer experienced in java spring bo...,188,181,False,NaN,NaN,NaN,NaN,NaN,NaN,backend engineer experienced in java spring bo...,0,0,0,0,0,0,181


,skill_id,skill_name,skill_type,group,relation_count,skill_group,skill_subgroup,mapped_taxonomy_group,mapped_taxonomy_subgroup,notes
0,NaN,3D lighting,essential,core,1,NaN,NaN,NaN,NaN,NaN
1,NaN,3D modelling,optional,core,1,NaN,NaN,NaN,NaN,NaN


,row_id,nhom_lon,nhom_nho,cum_ky_nang,skill_subgroup,ten_goc,ten_sach,ten_ngoai_thi_truong,cac_ten_gan_giong,huong_xu_ly,...,buoc,ten_thi_truong,ten_gan_giong,flag_missing_market_name,flag_missing_alias,market_name_norm,flag_duplicate_market_name,review_priority,review_issue,can_xem_thu_cong
0,2,Data & AI,AI / Machine Learning,AI / Data Tool,Machine Learning / AI,Python (computer programming),python (computer programming),python (computer programming),NaN,giu_nguyen,...,4,python (computer programming),NaN,0,0,python (computer programming),0,NaN,NaN,0
1,3,Data & AI,AI / Machine Learning,AI / Data Tool,Machine Learning / AI,computer vision,computer vision,computer vision,NaN,giu_nguyen,...,4,computer vision,NaN,0,0,computer vision,0,NaN,NaN,0


## 4. Kiểm tra cột bắt buộc trong CV

File `02_cv_sectioned.xlsx` cần có:

- `candidate_id`: mã ứng viên.
- Các section của CV như `section_skills`, `section_experience`, `section_projects`, `section_other`.

Nếu một số section không tồn tại, notebook vẫn có thể chạy bằng cách chỉ scan các section đang có.

In [21]:
required_cols = ["candidate_id"]

missing_cols = [col for col in required_cols if col not in cv_df.columns]
if missing_cols:
    raise ValueError(f"Thiếu cột bắt buộc trong CV section file: {missing_cols}")

candidate_text_sections = [
    col for col in [
        "section_skills",
        "section_experience",
        "section_projects",
        "section_other",
        "section_summary",
        "cv_text_clean",
    ]
    if col in cv_df.columns
]

print("Các section sẽ được scan skill:")
print(candidate_text_sections)

if not candidate_text_sections:
    raise ValueError("Không tìm thấy section text nào để scan skill.")

Các section sẽ được scan skill:
['section_skills', 'section_experience', 'section_projects', 'section_other', 'section_summary', 'cv_text_clean']


## 5. Hàm chuẩn hóa text

Mục đích: đưa text về dạng dễ so khớp hơn.

Ví dụ:

```text
"MySQL"      → "mysql"
" ReactJS "  → "reactjs"
"JavaScript!" → "javascript"
```

Nếu không chuẩn hóa, máy có thể coi `MySQL` và `mysql` là hai chuỗi khác nhau.

In [22]:
def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text


def normalize_skill_phrase(text):
    text = normalize_text(text)
    text = re.sub(r"[^\w\s\+\#\.\-/]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

## 6. Tìm cột skill chuẩn trong file 12

File `12_skill_master.xlsx` có thể đặt tên cột khác nhau tùy phiên bản.

Notebook thử tìm cột tên skill theo thứ tự ưu tiên.

Ví dụ:

- `skill_name`
- `skill`
- `preferred_label`
- `ten_sach`
- `ten_goc`

In [23]:
possible_skill_cols = [
    "skill_name",
    "skill",
    "preferred_label",
    "ten_sach",
    "ten_goc",
    "skill_clean",
]

skill_name_col = next(
    (col for col in possible_skill_cols if col in skill_df.columns),
    None
)

print("skill_name_col =", skill_name_col)

if skill_name_col is None:
    raise ValueError("Không tìm thấy cột tên skill trong 12_skill_master.xlsx")

skill_name_col = skill_name


## 7. Tạo danh sách skill chuẩn từ file 12

Bước này tạo một tập skill chuẩn.

Ví dụ:

```text
mysql
react
typescript
python
sql
```

Tập này là nguồn đầu tiên để scan skill trong CV.

In [24]:
canonical_skills = set()

for skill in skill_df[skill_name_col].dropna():
    skill_norm = normalize_skill_phrase(skill)
    if skill_norm:
        canonical_skills.add(skill_norm)

print("Số skill chuẩn từ file 12:", len(canonical_skills))
list(sorted(canonical_skills))[:20]

Số skill chuẩn từ file 12: 1172


['3d lighting',
 '3d modelling',
 '3d printing process',
 '3d texturing',
 'abap',
 'absorb learning management systems',
 'accounting techniques',
 'acquire system component',
 'adapt developed game to the market',
 'adapt to changes in technological development plans',
 'adapt to changing situations',
 'address problems critically',
 'adjust ict system capacity',
 'administer ict system',
 'adobe illustrator',
 'adobe photoshop',
 'advertising techniques',
 'advice on security risk management',
 'advise client on technical possibilities',
 'advise on communication strategies']

## 8. Xác định cột alias trong file Djinni

File Djinni có thể chứa nhiều cột tên gọi khác nhau của skill.

Ví dụ:

- `ten_sach`: tên chuẩn.
- `ten_goc`: tên gốc.
- `ten_ngoai_thi_truong`: tên gọi ngoài thị trường.
- `cac_ten_gan_giong`: các tên gần giống.

Bước này tìm các cột đó để tạo dictionary alias.

In [25]:
possible_canonical_cols = [
    "ten_sach",
    "skill_clean",
    "skill_name",
    "ten_goc",
]

possible_alias_cols = [
    "ten_goc",
    "ten_ngoai_thi_truong",
    "cac_ten_gan_giong",
]

djinni_canonical_col = next(
    (col for col in possible_canonical_cols if col in djinni_df.columns),
    None
)

djinni_alias_cols = [
    col for col in possible_alias_cols
    if col in djinni_df.columns
]

print("djinni_canonical_col =", djinni_canonical_col)
print("djinni_alias_cols =", djinni_alias_cols)

djinni_canonical_col = ten_sach
djinni_alias_cols = ['ten_goc', 'ten_ngoai_thi_truong', 'cac_ten_gan_giong']


## 9. Hàm parse alias

Alias trong Excel có thể nằm trong một ô với nhiều format khác nhau.

Ví dụ:

```text
"reactjs; react.js, react js | react"
```

Hàm này tách thành list:

```python
["reactjs", "react.js", "react js", "react"]
```

Mục đích: mỗi alias phải trở thành một từ khóa riêng để scan CV.

In [26]:
def parse_alias_value(value):
    if pd.isna(value):
        return []

    value = str(value).strip()
    if not value:
        return []

    if value.startswith("[") and value.endswith("]"):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return [str(x).strip() for x in parsed if str(x).strip()]
        except Exception:
            pass

    parts = re.split(r"[;,|]", value)
    return [part.strip() for part in parts if part.strip()]

## 10. Tạo dictionary `alias_to_skill`

Đây là phần quan trọng.

`alias_to_skill` là dictionary dạng:

```text
alias/tên gọi khác → skill chuẩn
```

Ví dụ:

```text
reactjs → react
js → javascript
postgres → postgresql
ml → machine learning
```

Mục đích: nếu CV viết tên thị trường hoặc tên gần giống, hệ thống vẫn quy về skill chuẩn.

In [27]:
alias_to_skill = {}

# Đưa skill chuẩn từ file 12 vào dictionary.
# Skill chuẩn cũng được xem là alias của chính nó.
for skill in canonical_skills:
    alias_to_skill[skill] = skill

# Đưa alias từ Djinni vào dictionary.
if djinni_canonical_col is not None:
    for _, row in djinni_df.iterrows():
        canonical_raw = row.get(djinni_canonical_col, "")
        canonical_norm = normalize_skill_phrase(canonical_raw)

        if not canonical_norm:
            continue

        alias_to_skill[canonical_norm] = canonical_norm

        for alias_col in djinni_alias_cols:
            alias_values = parse_alias_value(row.get(alias_col, ""))

            for alias in alias_values:
                alias_norm = normalize_skill_phrase(alias)
                if alias_norm:
                    alias_to_skill[alias_norm] = canonical_norm

print("Tổng số alias/skill phrase để scan:", len(alias_to_skill))
list(alias_to_skill.items())[:20]

Tổng số alias/skill phrase để scan: 1370


[('develop travel charter programme', 'develop travel charter programme'),
 ('use cam software', 'use cam software'),
 ('test ict queries', 'test ict queries'),
 ('apply control process statistical methods',
  'apply control process statistical methods'),
 ('finish processing of man-made fibres',
  'finish processing of man-made fibres'),
 ('maintain database performance', 'maintain database performance'),
 ('data quality assessment', 'data quality assessment'),
 ('handle game complaints', 'handle game complaints'),
 ('optimise choice of ict solution', 'optimise choice of ict solution'),
 ('perform image editing', 'perform image editing'),
 ('follow health and safety procedures in construction',
  'follow health and safety procedures in construction'),
 ('product life-cycle', 'product life-cycle'),
 ('distributed ledger technologies vulnerabilities',
  'distributed ledger technologies vulnerabilities'),
 ('use e-tourism platforms', 'use e-tourism platforms'),
 ('business ict systems', 

## 11. Sắp xếp phrase để scan

Notebook sẽ scan skill theo phrase dài trước.

Ví dụ nên scan:

```text
machine learning
```

trước:

```text
learning
```

Mục tiêu là giảm match nhầm.

In [28]:
skill_phrases_sorted = sorted(
    alias_to_skill.keys(),
    key=lambda phrase: len(phrase),
    reverse=True,
)

print("Số phrase scan:", len(skill_phrases_sorted))
print(skill_phrases_sorted[:30])

Số phrase scan: 1370
['apply research ethics and scientific integrity principles in research activities', 'promote the participation of citizens in scientific and research activities', 'engage local communities in the management of natural protected areas', 'interact professionally in research and professional environments', 'communicate commercial and technical issues in foreign languages', 'draft scientific or academic papers and technical documentation', 'improve customer traveling experiences with augmented reality', 'develop professional network with researchers and scientists', 'tourist resources of a destination for further development', 'manage findable accessible interoperable and reusable data', 'draw sketches to develop textile articles using softwares', 'manage distribution of destination promotional materials', 'organise participation in local or international events', 'assemble health and safety resources in cultural venues', 'integrate marketing strategies with the globa

## 12. Hàm extract skill từ một đoạn text

Hàm này nhận một đoạn text, rồi kiểm tra xem skill/alias nào xuất hiện trong text.

Ví dụ:

```text
Text: "I use reactjs, mysql and typescript"
```

Nếu dictionary có:

```text
reactjs → react
mysql → mysql
typescript → typescript
```

Kết quả:

```python
["react", "mysql", "typescript"]
```

In [29]:
def extract_skills_from_text(text):
    text_norm = normalize_skill_phrase(text)

    if not text_norm:
        return []

    matched = []

    for phrase in skill_phrases_sorted:
        # Match theo ranh giới từ để tránh match nhầm bên trong từ khác.
        pattern = r"(?<!\w)" + re.escape(phrase) + r"(?!\w)"
        if re.search(pattern, text_norm):
            matched.append(alias_to_skill[phrase])

    # Loại trùng nhưng giữ thứ tự xuất hiện trong quá trình scan.
    seen = set()
    result = []
    for skill in matched:
        if skill not in seen:
            seen.add(skill)
            result.append(skill)

    return result

## 13. Extract skill theo từng section CV

Với mỗi ứng viên, notebook scan từng section:

- `section_skills`
- `section_experience`
- `section_projects`
- `section_other`
- các section khác nếu có

Mỗi section sẽ có một cột kết quả riêng.

Ví dụ:

```text
matched_skills_experience_section = ["mysql", "java"]
```

In [30]:
cv_result_df = cv_df.copy()

section_output_cols = {}

for section_col in candidate_text_sections:
    output_col = "matched_skills_" + section_col.replace("section_", "").replace("cv_text_clean", "clean_text")
    section_output_cols[section_col] = output_col

    cv_result_df[output_col] = cv_result_df[section_col].apply(extract_skills_from_text)

print("Các cột skill theo section:")
print(section_output_cols)

preview_cols = ["candidate_id"] + list(section_output_cols.values())
display(cv_result_df[preview_cols].head())

Các cột skill theo section:
{'section_skills': 'matched_skills_skills', 'section_experience': 'matched_skills_experience', 'section_projects': 'matched_skills_projects', 'section_other': 'matched_skills_other', 'section_summary': 'matched_skills_summary', 'cv_text_clean': 'matched_skills_clean_text'}


,candidate_id,matched_skills_skills,matched_skills_experience,matched_skills_projects,matched_skills_other,matched_skills_summary,matched_skills_clean_text
0,C001,[],[],[],"[javascript, implement frontend website design...",[],"[javascript, implement frontend website design..."
1,C002,[],[],[],"[perform software unit testing, mysql]",[],"[perform software unit testing, mysql]"
2,C003,[],[],[],"[perform data analysis, sql]",[],"[perform data analysis, sql]"
3,C004,[],[],[],"[web services, devops]",[],"[web services, devops]"
4,C005,[],[],[],[],[],[]


## 14. Gộp skill từ tất cả section

Một skill có thể xuất hiện ở nhiều section.

Ví dụ:

```text
section_skills: ["mysql", "react"]
section_projects: ["react", "typescript"]
```

Sau khi gộp:

```text
matched_skills_all = ["mysql", "react", "typescript"]
```

Cột `n_matched_skills` là số lượng skill đã tìm được.

In [31]:
def merge_skill_lists(row, skill_cols):
    merged = []
    seen = set()

    for col in skill_cols:
        skills = row.get(col, [])
        if not isinstance(skills, list):
            skills = []

        for skill in skills:
            if skill not in seen:
                seen.add(skill)
                merged.append(skill)

    return merged


skill_result_cols = list(section_output_cols.values())

cv_result_df["matched_skills_all"] = cv_result_df.apply(
    lambda row: merge_skill_lists(row, skill_result_cols),
    axis=1,
)

cv_result_df["n_matched_skills"] = cv_result_df["matched_skills_all"].apply(len)

display(cv_result_df[[
    "candidate_id",
    "matched_skills_all",
    "n_matched_skills",
]].head())

,candidate_id,matched_skills_all,n_matched_skills
0,C001,"[javascript, implement frontend website design...",3
1,C002,"[perform software unit testing, mysql]",2
2,C003,"[perform data analysis, sql]",2
3,C004,"[web services, devops]",2
4,C005,[],0


## 15. Xem nhanh kết quả

Bước này chỉ dùng để kiểm tra CV nào extract được nhiều skill, CV nào extract được ít skill.

Nếu extract được ít skill, có thể do:

- CV quá ngắn.
- CV thiếu section rõ ràng.
- Skill master chưa đủ.
- Alias chưa đủ.
- Cách viết skill trong CV khác với dictionary hiện có.

In [32]:
preview_cols = ["candidate_id"]

for col in [
    "section_skills",
    "section_experience",
    "section_projects",
    "section_other",
]:
    if col in cv_result_df.columns:
        preview_cols.append(col)

preview_cols += ["matched_skills_all", "n_matched_skills"]

display(cv_result_df[preview_cols].head(10))

,candidate_id,section_skills,section_experience,section_projects,section_other,matched_skills_all,n_matched_skills
0,C001,NaN,NaN,NaN,frontend developer experienced in reactjs vuej...,"[javascript, implement frontend website design...",3
1,C002,NaN,NaN,NaN,backend engineer experienced in java spring bo...,"[perform software unit testing, mysql]",2
2,C003,NaN,NaN,NaN,data analyst experienced in python pandas nump...,"[perform data analysis, sql]",2
3,C004,NaN,NaN,NaN,devops engineer experienced in linux docker je...,"[web services, devops]",2
4,C005,NaN,NaN,NaN,qa engineer experienced in manual testing test...,[],0
5,C006,NaN,NaN,NaN,mobile developer experienced in flutter dart f...,[],0
6,C007,NaN,NaN,NaN,ai engineer experienced in python scikit-learn...,[machine learning],1
7,C008,NaN,NaN,NaN,cybersecurity analyst experienced in linux sec...,"[ict network security risks, cyber security]",2
8,C009,NaN,NaN,NaN,business analyst experienced in requirement ga...,[sql],1
9,C010,NaN,NaN,NaN,fullstack developer experienced in nestjs expr...,"[postgresql, typescript]",2


## 16. Lưu output

Xuất file:

```text
03_cv_skills_extracted.xlsx
```

File này là input cho bước 04.

Bước 04 sẽ lấy `matched_skills_all` rồi gán taxonomy cho từng skill.

In [33]:
cv_result_df.to_excel(OUTPUT_PATH, index=False)
print("Đã lưu file:", OUTPUT_PATH)

Đã lưu file: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ProcessPipeline/data_outputs/step03_skill_extract/03_cv_skills_extracted.xlsx


## 17. Đọc lại output để kiểm tra

In [34]:
result = pd.read_excel(OUTPUT_PATH, engine="openpyxl")

print("Output shape:", result.shape)
display(result.head())

Output shape: (20, 28)


,candidate_id,cv_text_raw,cv_text_clean,raw_length,clean_length,is_empty_clean,section_summary,section_skills,section_experience,section_education,...,section_certifications_len,section_other_len,matched_skills_skills,matched_skills_experience,matched_skills_projects,matched_skills_other,matched_skills_summary,matched_skills_clean_text,matched_skills_all,n_matched_skills
0,C001,"Frontend developer experienced in ReactJS, Vue...",frontend developer experienced in reactjs vuej...,249,240,False,NaN,NaN,NaN,NaN,...,0,240,[],[],[],"['javascript', 'implement frontend website des...",[],"['javascript', 'implement frontend website des...","['javascript', 'implement frontend website des...",3
1,C002,"Backend engineer experienced in Java, Spring B...",backend engineer experienced in java spring bo...,188,181,False,NaN,NaN,NaN,NaN,...,0,181,[],[],[],"['perform software unit testing', 'mysql']",[],"['perform software unit testing', 'mysql']","['perform software unit testing', 'mysql']",2
2,C003,"Data analyst experienced in Python, pandas, Nu...",data analyst experienced in python pandas nump...,163,154,False,NaN,NaN,NaN,NaN,...,0,154,[],[],[],"['perform data analysis', 'sql']",[],"['perform data analysis', 'sql']","['perform data analysis', 'sql']",2
3,C004,"DevOps engineer experienced in Linux, Docker, ...",devops engineer experienced in linux docker je...,173,165,False,NaN,NaN,NaN,NaN,...,0,165,[],[],[],"['web services', 'devops']",[],"['web services', 'devops']","['web services', 'devops']",2
4,C005,"QA engineer experienced in manual testing, tes...",qa engineer experienced in manual testing test...,172,166,False,NaN,NaN,NaN,NaN,...,0,166,[],[],[],[],[],[],[],0


## Tóm tắt file 03

```text
02_cv_sectioned.xlsx
+ 12_skill_master.xlsx
+ 08_djinni_step4_final.xlsx
        ↓
tạo alias_to_skill
        ↓
scan từng section CV
        ↓
matched_skills_all
        ↓
03_cv_skills_extracted.xlsx
```

Nói ngắn gọn:

> File 03 tạo một dictionary skill/alias, rồi dùng dictionary đó để quét CV và tìm ra các skill xuất hiện trong CV.